# Clasificación de prendas con Fashion MNIST (TensorFlow)
Cada bloque de código va precedido de la explicación de las funciones e instrucciones principales.

## 1. Importar librerías
- `numpy`: operaciones con arreglos numéricos (las imágenes son matrices).
- `matplotlib.pyplot`: gráficos e imágenes.
- `tensorflow`: librería de aprendizaje automático.
- `keras`: API de alto nivel de TensorFlow para construir redes neuronales.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

print("Versión de TensorFlow:", tf.__version__)

## 2. Descargar el dataset
- `keras.datasets.fashion_mnist.load_data()`: descarga Fashion MNIST y devuelve dos tuplas: (imágenes y etiquetas de entrenamiento) y (imágenes y etiquetas de prueba).
- Son 60.000 imágenes de entrenamiento y 10.000 de prueba, de 28x28 píxeles en escala de grises (valores 0 a 255).
- Las etiquetas son números del 0 al 9, por eso se define la lista `class_names` para saber qué prenda es cada número.
- `.shape`: dimensiones del arreglo.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

class_names = ['Camiseta/Top', 'Pantalón', 'Suéter', 'Vestido', 'Abrigo',
               'Sandalia', 'Camisa', 'Zapatilla', 'Bolso', 'Botín']

print("Entrenamiento:", x_train.shape, y_train.shape)
print("Prueba:", x_test.shape, y_test.shape)

## 3. Visualizar algunas imágenes
- `plt.figure(figsize=...)`: crea la figura con el tamaño indicado.
- `plt.subplot(filas, columnas, i)`: ubica cada imagen en una cuadrícula.
- `plt.imshow(img, cmap='gray')`: muestra la matriz como imagen en gris.
- `plt.title()`, `plt.axis('off')`: título y ocultar ejes.
- `plt.tight_layout()` / `plt.show()`: ajusta espacios y muestra el resultado.

In [ ]:
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap='gray')
    plt.title(class_names[y_train[i]], fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Preprocesamiento
- **Normalización** (`/ 255.0`): los píxeles pasan de [0, 255] a [0, 1]. Esto acelera y estabiliza el entrenamiento.
- **`np.expand_dims(..., -1)`**: agrega una dimensión de canal, de (28, 28) a (28, 28, 1). Las capas convolucionales esperan (alto, ancho, canales); aquí hay 1 canal porque es escala de grises.

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

print("Nueva forma:", x_train.shape)

## 5. Construir el modelo (red neuronal convolucional)
- `keras.Sequential([...])`: modelo formado por capas apiladas en orden.
- `keras.Input(shape=(28,28,1))`: define el tamaño de la entrada.
- `Conv2D(filtros, (3,3), activation='relu')`: aplica filtros de 3x3 que detectan patrones locales (bordes, texturas, formas). Cada filtro genera un mapa de características.
- `ReLU`: función de activación, deja pasar los valores positivos y anula los negativos; introduce no linealidad.
- `MaxPooling2D((2,2))`: reduce el tamaño a la mitad quedándose con el valor máximo de cada ventana 2x2; disminuye cálculos y da tolerancia a pequeños desplazamientos.
- `Flatten()`: convierte los mapas de características en un vector de una dimensión.
- `Dense(n, activation='relu')`: capa totalmente conectada de n neuronas.
- `Dropout(0.3)`: apaga aleatoriamente el 30% de las neuronas en cada paso de entrenamiento para evitar el sobreajuste.
- `Dense(10, activation='softmax')`: capa de salida, una neurona por clase; softmax convierte las salidas en probabilidades que suman 1.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    keras.layers.Conv2D(32, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(10, activation='softmax')
])

model.summary()

## 6. Compilar
- `model.compile()` configura el aprendizaje:
  - `optimizer='adam'`: algoritmo que ajusta los pesos de la red; adapta la tasa de aprendizaje automáticamente.
  - `loss='sparse_categorical_crossentropy'`: función de pérdida para clasificación multiclase cuando las etiquetas son enteros (0 a 9).
  - `metrics=['accuracy']`: métrica a reportar, la proporción de aciertos.
- `model.summary()` (celda anterior): muestra capas, formas de salida y número de parámetros.

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

## 7. Entrenar
- `model.fit()` entrena la red:
  - `epochs=10`: número de pasadas completas por el conjunto de entrenamiento.
  - `batch_size=64`: cantidad de imágenes procesadas antes de actualizar los pesos.
  - `validation_split=0.1`: separa el 10% de los datos para validar en cada época y detectar sobreajuste.
- Devuelve `history`, con la pérdida y precisión por época.

In [ ]:
history = model.fit(x_train, y_train,
                    epochs=10,
                    batch_size=64,
                    validation_split=0.1,
                    verbose=2)

## 8. Evaluar con el conjunto de prueba
- `model.evaluate(x_test, y_test)`: mide pérdida y precisión con datos que el modelo nunca vio. Es la medida real de su desempeño.

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Precisión en prueba: {test_acc*100:.2f}%")
print(f"Pérdida en prueba:   {test_loss:.4f}")
print(f"Precisión final entrenamiento: {history.history['accuracy'][-1]*100:.2f}%")
print(f"Precisión final validación:    {history.history['val_accuracy'][-1]*100:.2f}%")

## 9. Curvas de aprendizaje
- `history.history`: diccionario con `accuracy`, `val_accuracy`, `loss` y `val_loss` por época.
- Si la curva de entrenamiento sube pero la de validación se estanca o baja, hay sobreajuste.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['accuracy'], label='Entrenamiento')
ax[0].plot(history.history['val_accuracy'], label='Validación')
ax[0].set_title('Precisión'); ax[0].set_xlabel('Época'); ax[0].legend()
ax[1].plot(history.history['loss'], label='Entrenamiento')
ax[1].plot(history.history['val_loss'], label='Validación')
ax[1].set_title('Pérdida'); ax[1].set_xlabel('Época'); ax[1].legend()
plt.show()

## 10. Predicciones y análisis de errores
- `model.predict(x_test)`: devuelve, para cada imagen, 10 probabilidades (una por clase).
- `np.argmax(..., axis=1)`: toma la clase con mayor probabilidad como predicción.
- `classification_report`: precisión, exhaustividad (recall) y F1 por clase.
- `confusion_matrix`: tabla que muestra qué clases se confunden entre sí (filas = real, columnas = predicho).

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)

print(classification_report(y_test, y_pred, target_names=class_names, digits=3))

fig, ax = plt.subplots(figsize=(9, 9))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=class_names).plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.show()

In [ ]:
# Ejemplos de predicción (verde = acierto, rojo = error)
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap='gray')
    color = 'green' if y_pred[i] == y_test[i] else 'red'
    plt.title(f"Pred: {class_names[y_pred[i]]}\nReal: {class_names[y_test[i]]}",
              color=color, fontsize=8)
    plt.axis('off')
plt.tight_layout()
plt.show()

## Conclusiones

1. La CNN alcanzó una precisión en prueba de aproximadamente **91-92%** (reemplazar con el valor obtenido en la celda 8).
2. Las capas convolucionales y de pooling extraen patrones espaciales (bordes, formas), por lo que superan a una red densa simple.
3. Dropout ayudó a controlar el sobreajuste; la diferencia entre precisión de entrenamiento y validación se revisa en las curvas.
4. Las clases con más errores son camiseta, camisa, suéter y abrigo, por su parecido visual; las mejores son pantalón, bolso, botín y zapatilla.
5. Para mejorar: más épocas con early stopping, data augmentation, más filtros o arquitecturas más profundas.